# YOLO11n — Data Matrix locator (1 class)

**Zip:** `barcode_best_training_images.zip`  
**Goal:** Fine-tune YOLO11n to locate codes (mostly Data Matrix) for crop → ZXing decode.

| Item | Value |
|------|-------|
| Images | **349** (train **280** / val **69**) |
| Boxes | **934** |
| Class | **`0: datamatrix`** (renamed from `barcode`) |
| Model | `yolo11n.pt` fine-tune |
| imgsz / epochs / batch | **960** / **100** / **16** (T4) |
| Run name | `yolo11n_960_datamatrix_fresh` |

**Runtime → Change runtime type → T4 GPU** (or better)

> Class rename is label-only. Boxes stay class id `0`. This is a **locator**, not a format classifier — ZXing still decodes Data Matrix / QR / 1D.


## 1. GPU + install


In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


In [ ]:
!pip -q install -U ultralytics pyyaml
from ultralytics import YOLO
import ultralytics
print("ultralytics", ultralytics.__version__)


## 2. Upload dataset

Upload: **`barcode_best_training_images.zip`**  
(local path: `~/Downloads/barcode_best_training_images.zip`)

Or mount Drive and point `ZIP_PATH` at the zip (optional cell below).


In [ ]:
from google.colab import files
from pathlib import Path

ZIP_PATH = Path("/content/barcode_best_training_images.zip")
DATA_ROOT = Path("/content/barcode_best_training_images")

if not ZIP_PATH.exists() and not (DATA_ROOT / "data.yaml").exists():
    print("Select barcode_best_training_images.zip …")
    uploaded = files.upload()
    for name in uploaded:
        if name.endswith(".zip"):
            ZIP_PATH.write_bytes(uploaded[name])
            print("Saved", ZIP_PATH, ZIP_PATH.stat().st_size // (1024 * 1024), "MB")
            break
else:
    print("Zip or dataset already present")


In [ ]:
# Optional: load from Google Drive instead of upload
# from google.colab import drive
# from pathlib import Path
# drive.mount("/content/drive")
# ZIP_PATH = Path("/content/drive/MyDrive/barcode_best_training_images.zip")
# print(ZIP_PATH, ZIP_PATH.exists())


## 3. Unpack + set class name to `datamatrix`


In [ ]:
import zipfile
import collections
from pathlib import Path
import yaml

ZIP_PATH = Path("/content/barcode_best_training_images.zip")
DATA_ROOT = Path("/content/barcode_best_training_images")

assert ZIP_PATH.exists() or (DATA_ROOT / "data.yaml").exists(), "Upload zip first"

if not (DATA_ROOT / "images" / "train").exists():
    print("Unpacking…")
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(DATA_ROOT)

# Handle zip that nests an extra folder
if not (DATA_ROOT / "data.yaml").exists():
    nested = list(DATA_ROOT.rglob("data.yaml"))
    assert nested, "data.yaml not found after unzip"
    # if yaml is inside a subfolder with images/labels, retarget DATA_ROOT
    candidate = nested[0].parent
    if (candidate / "images").exists() or (candidate / "labels").exists():
        DATA_ROOT = candidate
        print("Using nested root:", DATA_ROOT)

yaml_path = DATA_ROOT / "data.yaml"
assert yaml_path.exists(), "data.yaml missing"

raw = yaml.safe_load(yaml_path.read_text()) or {}
# Force absolute Colab path + single class name
raw["path"] = str(DATA_ROOT)
raw["train"] = "images/train"
raw["val"] = "images/val"
raw["names"] = {0: "datamatrix"}
# Drop nc if present; Ultralytics infers from names
raw.pop("nc", None)

yaml_path.write_text(yaml.dump(raw, sort_keys=False, default_flow_style=False))
print("Wrote", yaml_path)
print(yaml_path.read_text())

# Sanity: class ids in labels
hist = collections.Counter()
n_empty = 0
label_files = list((DATA_ROOT / "labels").rglob("*.txt"))
for lf in label_files:
    lines = [ln.strip() for ln in lf.read_text().splitlines() if ln.strip()]
    if not lines:
        n_empty += 1
        continue
    for ln in lines:
        hist[int(ln.split()[0])] += 1

train_imgs = list((DATA_ROOT / "images" / "train").glob("*"))
val_imgs = list((DATA_ROOT / "images" / "val").glob("*"))
print(f"images train/val: {len(train_imgs)} / {len(val_imgs)}")
print(f"label files: {len(label_files)} (empty: {n_empty})")
print("class id histogram:", dict(sorted(hist.items())))
assert set(hist) <= {0}, f"Expected only class 0, got {dict(hist)}"
print("OK — ready to train")

# Persist root for later cells
Path("/content/DATA_ROOT.txt").write_text(str(DATA_ROOT))
print("DATA_ROOT saved for later cells")


## 4. Train YOLO11n


In [ ]:
from ultralytics import YOLO
from pathlib import Path

DATA_ROOT = Path(Path("/content/DATA_ROOT.txt").read_text().strip())
DATA_YAML = DATA_ROOT / "data.yaml"
assert DATA_YAML.exists(), DATA_YAML

PROJECT = "/content/runs/barcode"
RUN_NAME = "yolo11n_960_datamatrix_fresh"
IMGSZ = 960
EPOCHS = 100
BATCH = 16  # drop to 8 if OOM on T4

model = YOLO("yolo11n.pt")
results = model.train(
    data=str(DATA_YAML),
    imgsz=IMGSZ,
    epochs=EPOCHS,
    batch=BATCH,
    project=PROJECT,
    name=RUN_NAME,
    exist_ok=True,
    device=0,
)
print("Done. best.pt under", Path(PROJECT) / RUN_NAME / "weights")


## 5. Validate + find best weights


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/runs/barcode")
candidates = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)
assert candidates, "No best.pt found — did training finish?"
BEST = candidates[0]
print("Using", BEST)

DATA_YAML = Path(Path("/content/DATA_ROOT.txt").read_text().strip()) / "data.yaml"
model = YOLO(str(BEST))
metrics = model.val(data=str(DATA_YAML), imgsz=960)
print(metrics)


## 6. Export ONNX (for scansdk)


In [ ]:
from pathlib import Path
from ultralytics import YOLO

PROJECT = Path("/content/runs/barcode")
BEST = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
print("Exporting", BEST)

model = YOLO(str(BEST))
out = model.export(format="onnx", imgsz=960, simplify=True)
print("ONNX:", out)

# Copy with app-friendly name
onnx_src = Path(out)
onnx_dst = Path("/content/barcode-yolo11n.onnx")
onnx_dst.write_bytes(onnx_src.read_bytes())
print("Also wrote", onnx_dst, onnx_dst.stat().st_size // 1024, "KB")


## 7. Download weights

Download **`best.pt`** and **`barcode-yolo11n.onnx`**, then in the repo:

- `best.pt` → keep under `runs/.../weights/` (local)
- ONNX → `public/models/barcode-yolo11n.onnx` (app loads `/models/barcode-yolo11n.onnx`)


In [ ]:
from google.colab import files
from pathlib import Path

PROJECT = Path("/content/runs/barcode")
BEST = sorted(PROJECT.glob("**/weights/best.pt"), key=lambda p: p.stat().st_mtime, reverse=True)[0]
ONNX = Path("/content/barcode-yolo11n.onnx")
if not ONNX.exists():
    # fallback: exported next to best.pt
    siblings = list(BEST.parent.glob("*.onnx"))
    assert siblings, "ONNX missing — run export cell"
    ONNX = siblings[0]

print("Downloading", BEST)
files.download(str(BEST))
print("Downloading", ONNX)
files.download(str(ONNX))


## Tips

- **OOM:** set `BATCH = 8` (or `4`) and re-run train cell.
- **Runtime restart:** `/content` is wiped — re-upload zip + re-run install/unpack/train, or save run to Drive mid-training.
- **Class name:** only `datamatrix` — do not add empty QR/1D classes without labels.
- App decode still uses ZXing (`DataMatrix` preferred); YOLO only localizes boxes.
